<a href="https://colab.research.google.com/github/Karthikdude/testrepo/blob/main/Z_Image_Turbo_ComfyUI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Z-Image-Turbo GGUF — ComfyUI on Google Colab T4

**Runtime required:** GPU → T4 (16 GB VRAM)

| Component | Model | VRAM |
|---|---|---|
| UNet | Z-Image-Turbo Q5_K_M GGUF | ~5.5 GB |
| Text Encoder | Qwen3-4B Q4_K_M GGUF | ~2.5 GB |
| VAE | ae.safetensors | ~0.3 GB |
| PyTorch overhead | — | ~2.5 GB |
| **Total** | | **~10.8 GB ✅** |

**Run cells 1 → 5 in order. Cell 5 prints your public Cloudflare URL.**

## Cell 1 — GPU Check + System Dependencies

In [1]:
import subprocess, os

# Verify GPU
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)

# Install cloudflared
print('Installing cloudflared...')
os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb')
os.system('dpkg -i cloudflared-linux-amd64.deb -q')
os.system('rm -f cloudflared-linux-amd64.deb')
print('cloudflared installed.')

# Fast HF downloads
os.system('pip install -q huggingface_hub[hf_transfer]')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('huggingface_hub ready.')

Mon Mar  9 07:41:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Clone ComfyUI + Install GGUF Nodes

In [2]:
import os

COMFY = '/content/ComfyUI'

# ComfyUI
if not os.path.exists(COMFY):
    os.system(f'git clone -q https://github.com/comfyanonymous/ComfyUI {COMFY}')
else:
    os.system(f'cd {COMFY} && git pull -q')
os.system(f'pip install -q -r {COMFY}/requirements.txt')
print('ComfyUI ready.')

# ComfyUI-GGUF (city96) - required for .gguf model loading
GGUF = f'{COMFY}/custom_nodes/ComfyUI-GGUF'
if not os.path.exists(GGUF):
    os.system(f'git clone -q https://github.com/city96/ComfyUI-GGUF {GGUF}')
    os.system(f'pip install -q -r {GGUF}/requirements.txt')
else:
    os.system(f'cd {GGUF} && git pull -q')
print('ComfyUI-GGUF ready.')

# ComfyUI-Manager
MGR = f'{COMFY}/custom_nodes/ComfyUI-Manager'
if not os.path.exists(MGR):
    os.system(f'git clone -q https://github.com/ltdrdata/ComfyUI-Manager {MGR}')
else:
    os.system(f'cd {MGR} && git pull -q')
print('ComfyUI-Manager ready.')
print('All nodes installed.')

ComfyUI ready.
ComfyUI-GGUF ready.
ComfyUI-Manager ready.
All nodes installed.


## Cell 3 — Download Models

Downloads ~8.4 GB total. Takes 5-10 min on Colab. Skip if already downloaded this session.

In [4]:
import os
from huggingface_hub import login, hf_hub_download

# ================================
# 1. Login to HuggingFace
# ================================
login(token="")

COMFY = "/content/ComfyUI"

# ================================
# 2. Create required folders
# ================================
for d in ["unet", "text_encoders", "vae"]:
    os.makedirs(f"{COMFY}/models/{d}", exist_ok=True)

# ================================
# 3. Download Z-Image-Turbo UNet
# ================================
UNET = f"{COMFY}/models/unet/z_image_turbo-Q5_K_M.gguf"

if not os.path.exists(UNET):
    print("Downloading Z-Image-Turbo Q5_K_M (~5.5GB)...")
    hf_hub_download(
        repo_id="jayn7/Z-Image-Turbo-GGUF",
        filename="z_image_turbo-Q5_K_M.gguf",
        local_dir=f"{COMFY}/models/unet",
        local_dir_use_symlinks=False
    )

print("Z-Image-Turbo UNet ready.")

# ================================
# 4. Download Qwen3 Text Encoder
# ================================
QWEN = f"{COMFY}/models/text_encoders/Qwen3-4B-Q4_K_M.gguf"

if not os.path.exists(QWEN):
    print("Downloading Qwen3-4B Text Encoder (~2.5GB)...")
    hf_hub_download(
        repo_id="unsloth/Qwen3-4B-GGUF",
        filename="Qwen3-4B-Q4_K_M.gguf",
        local_dir=f"{COMFY}/models/text_encoders",
        local_dir_use_symlinks=False
    )

print("Qwen3 Text Encoder ready.")

# ================================
# 5. Download FLUX VAE (Gated)
# ================================
VAE = f"{COMFY}/models/vae/ae.safetensors"

if not os.path.exists(VAE):
    print("Downloading FLUX VAE (~335MB)...")
    hf_hub_download(
        repo_id="black-forest-labs/FLUX.1-dev",
        filename="ae.safetensors",
        local_dir=f"{COMFY}/models/vae",
        local_dir_use_symlinks=False
    )

print("FLUX VAE ready.")

# ================================
# 6. Verify files
# ================================
print("\nDownloaded file sizes:")

for path, label in [
    (UNET, "UNet"),
    (QWEN, "Text Encoder"),
    (VAE, "VAE")
]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e9
        print(f"[OK] {label}: {size:.2f} GB")
    else:
        print(f"[MISSING] {label}")

Z-Image-Turbo UNet ready.
Qwen3 Text Encoder ready.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


ae.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

FLUX VAE ready.

Downloaded file sizes:
[OK] UNet: 5.52 GB
[OK] Text Encoder: 2.50 GB
[OK] VAE: 0.34 GB


## Cell 4 — Write ComfyUI Workflow

In [5]:
import json, os

COMFY = '/content/ComfyUI'

workflow = {
  'last_node_id': 9,
  'last_link_id': 9,
  'nodes': [
    {'id':1,'type':'UnetLoaderGGUF','pos':[50,100],'size':{'0':300,'1':58},'flags':{},'order':0,'mode':0,
     'outputs':[{'name':'MODEL','type':'MODEL','links':[1],'slot_index':0}],
     'properties':{},'widgets_values':['z_image_turbo-Q5_K_M.gguf']},
    {'id':2,'type':'CLIPLoaderGGUF','pos':[50,200],'size':{'0':300,'1':58},'flags':{},'order':1,'mode':0,
     'outputs':[{'name':'CLIP','type':'CLIP','links':[2,3],'slot_index':0}],
     'properties':{},'widgets_values':['Qwen3-4B-Q4_K_M.gguf','lumina2']},
    {'id':3,'type':'VAELoader','pos':[50,300],'size':{'0':300,'1':58},'flags':{},'order':2,'mode':0,
     'outputs':[{'name':'VAE','type':'VAE','links':[4],'slot_index':0}],
     'properties':{},'widgets_values':['ae.safetensors']},
    {'id':4,'type':'CLIPTextEncode','pos':[400,100],'size':{'0':400,'1':120},'flags':{},'order':3,'mode':0,
     'inputs':[{'name':'clip','type':'CLIP','link':2}],
     'outputs':[{'name':'CONDITIONING','type':'CONDITIONING','links':[5],'slot_index':0}],
     'properties':{},'widgets_values':['A breathtaking photorealistic portrait of a young woman in a sunlit Japanese garden, cherry blossom petals falling gently, soft white linen dress, golden hour light, bokeh background, ultra-detailed skin, 8K']},
    {'id':5,'type':'CLIPTextEncode','pos':[400,260],'size':{'0':400,'1':80},'flags':{},'order':4,'mode':0,
     'inputs':[{'name':'clip','type':'CLIP','link':3}],
     'outputs':[{'name':'CONDITIONING','type':'CONDITIONING','links':[6],'slot_index':0}],
     'properties':{},'widgets_values':['blurry, low quality, watermark, text, ugly, deformed, bad anatomy']},
    {'id':6,'type':'EmptyLatentImage','pos':[400,370],'size':{'0':300,'1':106},'flags':{},'order':5,'mode':0,
     'outputs':[{'name':'LATENT','type':'LATENT','links':[7],'slot_index':0}],
     'properties':{},'widgets_values':[1024,1024,1]},
    {'id':7,'type':'KSampler','pos':[860,180],'size':{'0':315,'1':262},'flags':{},'order':6,'mode':0,
     'inputs':[
       {'name':'model','type':'MODEL','link':1},
       {'name':'positive','type':'CONDITIONING','link':5},
       {'name':'negative','type':'CONDITIONING','link':6},
       {'name':'latent_image','type':'LATENT','link':7}
     ],
     'outputs':[{'name':'LATENT','type':'LATENT','links':[8],'slot_index':0}],
     'properties':{},'widgets_values':[42,'fixed',8,3.5,'dpmpp_2m','karras',1.0]},
    {'id':8,'type':'VAEDecode','pos':[1230,260],'size':{'0':210,'1':46},'flags':{},'order':7,'mode':0,
     'inputs':[
       {'name':'samples','type':'LATENT','link':8},
       {'name':'vae','type':'VAE','link':4}
     ],
     'outputs':[{'name':'IMAGE','type':'IMAGE','links':[9],'slot_index':0}],
     'properties':{}},
    {'id':9,'type':'SaveImage','pos':[1480,200],'size':{'0':210,'1':58},'flags':{},'order':8,'mode':0,
     'inputs':[{'name':'images','type':'IMAGE','link':9}],
     'outputs':[],'properties':{},'widgets_values':['z_image_turbo']}
  ],
  'links':[
    [1,1,0,7,0,'MODEL'],
    [2,2,0,4,0,'CLIP'],
    [3,2,0,5,0,'CLIP'],
    [4,3,0,8,1,'VAE'],
    [5,4,0,7,1,'CONDITIONING'],
    [6,5,0,7,2,'CONDITIONING'],
    [7,6,0,7,3,'LATENT'],
    [8,7,0,8,0,'LATENT'],
    [9,8,0,9,0,'IMAGE']
  ],
  'groups':[],'config':{},'extra':{},'version':0.4
}

wf_dir = f'{COMFY}/user/default/workflows'
os.makedirs(wf_dir, exist_ok=True)
with open(f'{wf_dir}/Z_Image_Turbo_GGUF.json', 'w') as f:
    json.dump(workflow, f, indent=2)
print('Workflow saved. In ComfyUI: Workflow -> Open -> Z_Image_Turbo_GGUF')

Workflow saved. In ComfyUI: Workflow -> Open -> Z_Image_Turbo_GGUF


## Cell 5 — Launch ComfyUI with Cloudflare Tunnel

**Keep this cell running.** The public URL prints once ComfyUI is ready (~30s).

In [ ]:
import subprocess, threading, time, socket, os, shutil, re

COMFY = '/content/ComfyUI'
PORT = 8188

# ── Install cloudflared if missing ──────────────────────────────────────────
if not shutil.which('cloudflared'):
    print('Installing cloudflared...')
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
    print('cloudflared installed ✓')
else:
    print('cloudflared already installed ✓')

# ── Tunnel thread ────────────────────────────────────────────────────────────
def launch_tunnel(port):
    print(f'Waiting for ComfyUI on port {port}...')
    for _ in range(120):
        time.sleep(1)
        s = socket.socket()
        r = s.connect_ex(('127.0.0.1', port))
        s.close()
        if r == 0:
            break

    print('ComfyUI is up. Starting Cloudflare tunnel...')
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT  # merge both streams
    )
    for line in proc.stdout:
        d = line.decode()
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', d)
        if match:
            url = match.group(0)
            print('=' * 55)
            print('  ComfyUI PUBLIC URL:')
            print(f'  {url}')
            print('=' * 55)
            break

threading.Thread(target=launch_tunnel, args=(PORT,), daemon=True).start()

# ── Launch ComfyUI (blocks until server stops) ───────────────────────────────
os.chdir(COMFY)
os.system(
    f'python main.py --listen 0.0.0.0 --port {PORT} '
    f'--highvram --cuda-malloc --dont-print-server'
)

cloudflared already installed ✓
Waiting for ComfyUI on port 8188...
ComfyUI is up. Starting Cloudflare tunnel...
  ComfyUI PUBLIC URL:
  https://dreams-exchanges-bid-dependent.trycloudflare.com


## Cell 6 (Optional) — Headless Diffusers Inference

> Only run if Cell 5 (ComfyUI) is NOT running — both together will OOM.

In [ ]:
# Stop ComfyUI first, then run this cell
import os
os.system('pip install -q diffusers>=0.34.0 accelerate')

import torch
from diffusers import ZImagePipeline, ZImageTransformer2DModel, GGUFQuantizationConfig
from IPython.display import display

COMFY = '/content/ComfyUI'
GGUF_PATH = f'{COMFY}/models/unet/z_image_turbo-Q5_K_M.gguf'

print('Loading GGUF transformer...')
transformer = ZImageTransformer2DModel.from_single_file(
    GGUF_PATH,
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    dtype=torch.bfloat16,
)

print('Building pipeline...')
pipe = ZImagePipeline.from_pretrained(
    'Tongyi-MAI/Z-Image-Turbo',
    transformer=transformer,
    torch_dtype=torch.bfloat16,
).to('cuda')

PROMPT = 'A breathtaking photorealistic portrait of a young woman in a sunlit Japanese garden, cherry blossom petals falling gently, soft white linen dress, golden hour light, bokeh background, ultra-detailed skin, 8K'
NEG = 'blurry, low quality, watermark, ugly, deformed'

print('Generating image (8 steps, ~15s on T4)...')
with torch.inference_mode():
    result = pipe(
        prompt=PROMPT,
        negative_prompt=NEG,
        height=1024, width=1024,
        num_inference_steps=8,
        guidance_scale=3.5,
        generator=torch.Generator('cuda').manual_seed(42),
    )

img = result.images[0]
img.save('/content/z_image_output.png')
display(img)
used = torch.cuda.memory_allocated() / 1e9
print(f'Done. VRAM used: {used:.2f} GB')
print('Saved to /content/z_image_output.png')

## Reference — GGUF Quant Levels for Z-Image-Turbo

| Quant | Size | Quality | Best For |
|-------|------|---------|----------|
| Q3_K_S | 3.79 GB | ⭐⭐ | 8 GB cards only |
| Q4_K_S | 4.66 GB | ⭐⭐⭐ | 12 GB VRAM |
| Q4_K_M | 4.98 GB | ⭐⭐⭐ | Good balance |
| **Q5_K_M** | **5.52 GB** | **⭐⭐⭐⭐** | **This notebook (T4 16 GB)** |
| Q6_K | 5.91 GB | ⭐⭐⭐⭐ | Also fits T4 (tight) |
| Q8_0 | 7.22 GB | ⭐⭐⭐⭐⭐ | 16 GB VRAM only |
| BF16 | 12.3 GB | ⭐⭐⭐⭐⭐ | Does not fit T4 |

To switch quants: edit the filename in **Cell 3** and **Cell 4** `widgets_values[0]`.

## Troubleshooting

| Problem | Fix |
|---------|-----|
| 502 on tunnel URL | Wait 30s and reload |
| CUDA OOM | Lower to 768x768 or use Q4_K_S |
| Red nodes in ComfyUI | Manager -> Install Missing Custom Nodes |
| Tunnel URL not printing | Re-run Cell 5 |
| CLIPLoaderGGUF not found | Re-run Cell 2 |